# Day 7: Leakage-Safe Baseline Modelling

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objective
Audit the Day 4 modelling tables, review feature eligibility, create chronological train/validation/test splits, and compare simple regression baselines.

**Targets:** `next_match_runs`, `next_match_wickets`, and `next_match_runs_conceded`.

**Leakage controls:** Do not use current-match outcomes as predictors; only numeric `prior_...` features are automatic candidates. Use chronological splits and fit imputation on training data only. Review generated feature and split reports before accepting results.


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (p / "data" / "processed" / "batting_next_match_features.csv").is_file() or (
        (p / "data" / "processed").is_dir() and (p / "notebooks").is_dir()
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "day7"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
BATTING_PATH = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_PATH = PROCESSED_DIR / "bowling_next_match_features.csv"
print("Working directory:", current_dir)
print("Project root:", PROJECT_ROOT)
print("Batting file:", "FOUND" if BATTING_PATH.is_file() else "MISSING")
print("Bowling file:", "FOUND" if BOWLING_PATH.is_file() else "MISSING")


Working directory: D:\Git\Personal\ipl-player-performance\notebooks
Project root: D:\Git\Personal\ipl-player-performance
Batting file: FOUND
Bowling file: FOUND


In [2]:
def read_dataset(path, required):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run and validate Day 4 first.")
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing columns {sorted(missing)}; found {list(df.columns)}")
    return df

batting = read_dataset(BATTING_PATH, {"player_name", "match_id", "date", "next_match_date", "next_match_runs"})
bowling = read_dataset(BOWLING_PATH, {"player_name", "match_id", "date", "next_match_date",
                                     "next_match_wickets", "next_match_runs_conceded"})
for name, df in [("batting", batting), ("bowling", bowling)]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["next_match_date"] = pd.to_datetime(df["next_match_date"], errors="coerce")
    print(f"{name}: {len(df):,} rows x {len(df.columns)} columns")
    print("Columns:", list(df.columns))


batting: 18,030 rows x 49 columns
Columns: ['match_id', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type', 'team', 'opposition', 'player_name', 'player_id', 'player_name_norm', 'team_norm', 'opposition_norm', 'runs', 'balls_faced', 'fours', 'sixes', 'batting_position', 'was_dismissed', 'dismissal_kind', 'innings_count', 'strike_rate', 'match_date', 'season_meta', 'venue_meta', 'city_meta', 'team_1_meta', 'team_2_meta', 'date', 'next_match_runs', 'next_match_id', 'next_match_date', 'matches_seen_before', 'prior_career_batting_runs', 'prior_last5_mean_batting_runs', 'prior_last5_count_batting_runs', 'prior_career_batting_balls_faced', 'prior_last5_mean_batting_balls_faced', 'prior_last5_count_batting_balls_faced', 'prior_career_batting_fours', 'prior_last5_mean_batting_fours', 'prior_last5_count_batting_fours', 'prior_career_batting_sixes', 'prior_last5_mean_batting_sixes', 'prior_last5_count_batting_sixes', 'prior_career_batting_strike_rate', 'prior_last5_mean_batting_strike_r

## 1. Data readiness and target audit

This section reports missing identifiers/dates, duplicate player-match keys, and target distributions. It does not silently delete or repair rows.


In [3]:
audit_rows, target_rows = [], []
for name, df, targets in [
    ("batting", batting, ["next_match_runs"]),
    ("bowling", bowling, ["next_match_wickets", "next_match_runs_conceded"]),
]:
    audit_rows.append({
        "dataset": name, "rows": len(df), "players": df["player_name"].nunique(),
        "missing_player": int(df["player_name"].isna().sum()),
        "missing_match_id": int(df["match_id"].isna().sum()),
        "invalid_current_date": int(df["date"].isna().sum()),
        "invalid_target_date": int(df["next_match_date"].isna().sum()),
        "duplicate_match_player_keys": int(df.duplicated(["match_id", "player_name"]).sum())
    })
    for target in targets:
        vals = pd.to_numeric(df[target], errors="coerce")
        target_rows.append({"dataset": name, "target": target, "non_null": int(vals.notna().sum()),
                            "missing": int(vals.isna().sum()), "mean": vals.mean(),
                            "median": vals.median(), "std": vals.std(), "min": vals.min(), "max": vals.max()})
audit_df = pd.DataFrame(audit_rows)
target_summary = pd.DataFrame(target_rows)
display(audit_df)
display(target_summary)
audit_df.to_csv(REPORT_DIR / "data_readiness_audit.csv", index=False)
target_summary.to_csv(REPORT_DIR / "target_summary.csv", index=False)


,dataset,rows,players,missing_player,missing_match_id,invalid_current_date,invalid_target_date,duplicate_match_player_keys
0,batting,18030,633,0,0,0,0,0
1,bowling,14123,503,0,0,0,0,0


,dataset,target,non_null,missing,mean,median,std,min,max
0,batting,next_match_runs,18030,0,20.704049,13.0,21.830644,0.0,175.0
1,bowling,next_match_wickets,14123,0,0.922396,1.0,1.001237,0.0,6.0
2,bowling,next_match_runs_conceded,14123,0,26.897260,26.0,11.188855,0.0,76.0


## 2. Feature eligibility review

Automatic predictors are restricted to numeric `prior_...` columns. Current-match outcome columns, target columns, identifiers, and future-match metadata are excluded. Categorical context columns are marked for manual review rather than silently encoded.


In [4]:
TARGETS = {"batting": ["next_match_runs"],
           "bowling": ["next_match_wickets", "next_match_runs_conceded"]}
IDENTIFIERS = {"player_name", "player_id", "match_id", "next_match_id", "date", "next_match_date", "source_file"}
CURRENT_OUTCOMES = {"runs", "balls_faced", "fours", "sixes", "strike_rate", "batting_position", "dismissed",
                    "wickets", "runs_conceded", "legal_balls", "overs", "economy", "maidens", "dots", "wides", "noballs"}
CONTEXT = {"season", "venue", "city", "team", "opposition", "team_1", "team_2"}

def review_features(df, name):
    rows = []
    for col in df.columns:
        if col in TARGETS[name]:
            decision, reason = "EXCLUDE", "target"
        elif col in IDENTIFIERS:
            decision, reason = "EXCLUDE", "identifier/date/future metadata"
        elif col in CURRENT_OUTCOMES:
            decision, reason = "EXCLUDE", "current-match outcome; potential leakage"
        elif col.startswith("prior_") and pd.api.types.is_numeric_dtype(df[col]):
            decision, reason = "ELIGIBLE_CANDIDATE", "numeric prior-history feature; verify construction"
        elif col in CONTEXT:
            decision, reason = "REVIEW_MANUALLY", "categorical context needs encoding and timing review"
        elif pd.api.types.is_numeric_dtype(df[col]):
            decision, reason = "REVIEW_MANUALLY", "numeric column outside prior-history whitelist"
        else:
            decision, reason = "EXCLUDE", "not an approved numeric predictor"
        rows.append({"dataset": name, "column": col, "dtype": str(df[col].dtype),
                     "decision": decision, "reason": reason, "missing_pct": round(df[col].isna().mean()*100, 2)})
    return pd.DataFrame(rows)

batting_review = review_features(batting, "batting")
bowling_review = review_features(bowling, "bowling")
feature_review_df = pd.concat([batting_review, bowling_review], ignore_index=True)
display(feature_review_df)
feature_review_df.to_csv(REPORT_DIR / "feature_eligibility_review.csv", index=False)
batting_features = batting_review.loc[batting_review.decision == "ELIGIBLE_CANDIDATE", "column"].tolist()
bowling_features = bowling_review.loc[bowling_review.decision == "ELIGIBLE_CANDIDATE", "column"].tolist()
print("Eligible batting features:", batting_features)
print("Eligible bowling features:", bowling_features)


,dataset,column,dtype,decision,reason,missing_pct
0,batting,match_id,int64,EXCLUDE,identifier/date/future metadata,0.00
1,batting,season,str,REVIEW_MANUALLY,categorical context needs encoding and timing ...,0.00
2,batting,venue,str,REVIEW_MANUALLY,categorical context needs encoding and timing ...,0.00
3,batting,city,str,REVIEW_MANUALLY,categorical context needs encoding and timing ...,4.08
4,batting,team_1,str,REVIEW_MANUALLY,categorical context needs encoding and timing ...,0.00
...,...,...,...,...,...,...
94,bowling,prior_last5_mean_bowling_dots,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify construc...,3.56
95,bowling,prior_last5_count_bowling_dots,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify construc...,0.00
96,bowling,prior_career_bowling_economy,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify construc...,3.56
97,bowling,prior_last5_mean_bowling_economy,float64,ELIGIBLE_CANDIDATE,numeric prior-history feature; verify construc...,3.56


Eligible batting features: ['prior_career_batting_runs', 'prior_last5_mean_batting_runs', 'prior_last5_count_batting_runs', 'prior_career_batting_balls_faced', 'prior_last5_mean_batting_balls_faced', 'prior_last5_count_batting_balls_faced', 'prior_career_batting_fours', 'prior_last5_mean_batting_fours', 'prior_last5_count_batting_fours', 'prior_career_batting_sixes', 'prior_last5_mean_batting_sixes', 'prior_last5_count_batting_sixes', 'prior_career_batting_strike_rate', 'prior_last5_mean_batting_strike_rate', 'prior_last5_count_batting_strike_rate']
Eligible bowling features: ['prior_career_bowling_wickets', 'prior_last5_mean_bowling_wickets', 'prior_last5_count_bowling_wickets', 'prior_career_bowling_runs_conceded', 'prior_last5_mean_bowling_runs_conceded', 'prior_last5_count_bowling_runs_conceded', 'prior_career_bowling_legal_balls', 'prior_last5_mean_bowling_legal_balls', 'prior_last5_count_bowling_legal_balls', 'prior_career_bowling_dots', 'prior_last5_mean_bowling_dots', 'prior_la

## 3. Chronological splits and baseline metrics

Each table is sorted by current feature-row date, match ID, and player name. The earliest 70% is training, the next 15% validation, and the latest 15% test. This is a row-based split; inspect boundaries and be aware that records from the same match could straddle a boundary.

Models: a median `DummyRegressor` and `LinearRegression`. Imputation is inside a pipeline and therefore fitted on training rows only. Metrics are MAE, RMSE, and R².


In [5]:
def chronological_split(df, target):
    work = df.loc[df[target].notna() & df["date"].notna()].copy()
    work = work.sort_values(["date", "match_id", "player_name"], kind="stable").reset_index(drop=True)
    if len(work) < 20:
        raise ValueError(f"{target}: only {len(work)} usable rows; at least 20 required for this split.")
    n = len(work)
    a, b = int(n * 0.70), int(n * 0.85)
    if a < 2 or b <= a or b >= n:
        raise ValueError(f"{target}: unable to form non-empty chronological splits.")
    return work, work.iloc[:a].copy(), work.iloc[a:b].copy(), work.iloc[b:].copy()

def evaluate_target(df, dataset_name, target, feature_cols):
    if not feature_cols:
        print(f"SKIPPED {dataset_name}/{target}: no approved prior-history features.")
        return [], None
    work, train, val, test = chronological_split(df, target)
    usable = [c for c in feature_cols if train[c].notna().any()]
    if not usable:
        print(f"SKIPPED {dataset_name}/{target}: all candidate features missing in training rows.")
        return [], None
    def matrix(part):
        return part[usable].apply(pd.to_numeric, errors="coerce")
    X_train, X_val, X_test = matrix(train), matrix(val), matrix(test)
    y_train = pd.to_numeric(train[target], errors="coerce")
    rows = []
    models = {
        "DummyMedian": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                 ("model", DummyRegressor(strategy="median"))]),
        "LinearRegression": Pipeline([("imputer", SimpleImputer(strategy="median")),
                                      ("model", LinearRegression())])
    }
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        for split_name, part, X in [("validation", val, X_val), ("test", test, X_test)]:
            y = pd.to_numeric(part[target], errors="coerce")
            pred = model.predict(X)
            rows.append({"dataset": dataset_name, "target": target, "model": model_name,
                         "split": split_name, "rows": len(part), "feature_count": len(usable),
                         "MAE": mean_absolute_error(y, pred),
                         "RMSE": np.sqrt(mean_squared_error(y, pred)),
                         "R2": r2_score(y, pred) if len(y) >= 2 else np.nan})
    split = {"dataset": dataset_name, "target": target, "total_rows": len(work),
             "train_rows": len(train), "validation_rows": len(val), "test_rows": len(test),
             "train_start": str(train.date.min().date()), "train_end": str(train.date.max().date()),
             "validation_start": str(val.date.min().date()), "validation_end": str(val.date.max().date()),
             "test_start": str(test.date.min().date()), "test_end": str(test.date.max().date()),
             "feature_columns": ", ".join(usable)}
    return rows, split

metric_rows, split_rows = [], []
for target in TARGETS["batting"]:
    rows, split = evaluate_target(batting, "batting", target, batting_features)
    metric_rows.extend(rows)
    if split: split_rows.append(split)
for target in TARGETS["bowling"]:
    rows, split = evaluate_target(bowling, "bowling", target, bowling_features)
    metric_rows.extend(rows)
    if split: split_rows.append(split)

metrics_df = pd.DataFrame(metric_rows)
splits_df = pd.DataFrame(split_rows)
display(splits_df)
display(metrics_df)
metrics_df.to_csv(REPORT_DIR / "baseline_model_metrics.csv", index=False)
splits_df.to_csv(REPORT_DIR / "chronological_split_summary.csv", index=False)


,dataset,target,total_rows,train_rows,validation_rows,test_rows,train_start,train_end,validation_start,validation_end,test_start,test_end,feature_columns
0,batting,next_match_runs,18030,12621,2704,2705,2008-04-18,2021-10-08,2021-10-08,2024-04-13,2024-04-13,2026-05-29,"prior_career_batting_runs, prior_last5_mean_ba..."
1,bowling,next_match_wickets,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,"prior_career_bowling_wickets, prior_last5_mean..."
2,bowling,next_match_runs_conceded,14123,9886,2118,2119,2008-04-18,2021-10-01,2021-10-01,2024-04-08,2024-04-08,2026-05-29,"prior_career_bowling_wickets, prior_last5_mean..."


,dataset,target,model,split,rows,feature_count,MAE,RMSE,R2
0,batting,next_match_runs,DummyMedian,validation,2704,15,16.475222,23.850519,-0.129310
1,batting,next_match_runs,DummyMedian,test,2705,15,17.912754,26.069777,-0.188754
2,batting,next_match_runs,LinearRegression,validation,2704,15,15.864845,21.231884,0.105058
3,batting,next_match_runs,LinearRegression,test,2705,15,16.835979,22.663426,0.101603
4,bowling,next_match_wickets,DummyMedian,validation,2118,15,0.789424,1.051551,-0.000545
5,bowling,next_match_wickets,DummyMedian,test,2119,15,0.759320,1.002828,-0.000410
6,bowling,next_match_wickets,LinearRegression,validation,2118,15,0.789086,1.036961,0.027026
7,bowling,next_match_wickets,LinearRegression,test,2119,15,0.767192,0.990279,0.024471
8,bowling,next_match_runs_conceded,DummyMedian,validation,2118,15,9.516997,11.945718,-0.089322
9,bowling,next_match_runs_conceded,DummyMedian,test,2119,15,10.924021,13.644409,-0.221064


## 4. Interpretation and next step

- **MAE:** average absolute prediction error, in runs or wickets.
- **RMSE:** gives larger errors more weight.
- **R²:** may be negative on unseen data.
- **DummyMedian:** simple benchmark.
- **LinearRegression:** first trainable regression baseline.

Use validation metrics for model selection. Keep the test set as a final holdout. Do not accept results until you inspect feature eligibility and date boundaries. This day does not train Random Forest, XGBoost, LSTM, or GRU.


In [6]:
readiness_text = audit_df.to_string(index=False)
target_text = target_summary.to_string(index=False)
feature_counts = feature_review_df.groupby(["dataset", "decision"]).size().unstack(fill_value=0).to_string()
split_text = splits_df.to_string(index=False) if not splits_df.empty else "No split summaries were produced."
metric_text = metrics_df.to_string(index=False) if not metrics_df.empty else "No baseline metrics were produced."

doc = f"""# Day 7: Leakage-Safe Baseline Modelling

## Data readiness audit
```text
{readiness_text}
```

## Target summary
```text
{target_text}
```

## Feature eligibility counts
```text
{feature_counts}
```

## Chronological split boundaries
```text
{split_text}
```

## Baseline metrics
```text
{metric_text}
```

## Outputs
- `reports/day7/data_readiness_audit.csv`
- `reports/day7/target_summary.csv`
- `reports/day7/feature_eligibility_review.csv`
- `reports/day7/chronological_split_summary.csv`
- `reports/day7/baseline_model_metrics.csv`

## Safeguards and limitations
- Current-match outcomes, targets, identifiers, and future-match metadata are excluded from automatic predictors.
- Automatic predictor candidates are numeric `prior_...` columns; inspect their construction in the Day 4 notebook.
- Chronological split is row-based (70%/15%/15%), so records from one match may cross a split boundary.
- Same-day ordering uses match ID as a tie-breaker where start times are unavailable.
- Imputation is fitted within each model pipeline on training data only.
- No metric is asserted until this notebook is run against the user's actual data.

## Next step
Day 8: train Random Forest and XGBoost regressors and compare them with these baselines.
"""
doc_path = DOCS_DIR / "day7_leakage_safe_baseline_modelling.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved documentation:", doc_path)


Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day7_leakage_safe_baseline_modelling.md
